# DATA ANALYSIS 
## DATA COVERAGE 
### 1. DATA CLEANING
#### a. UNSD Environmental Indicators Metadata and Country Availability Workbook


In [15]:
# ============================================================
# UNSD Environmental Indicators Metadata + Country Availability
# ============================================================

import os
import re
import time
import requests
import pandas as pd
import numpy as np

from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin, unquote, urlparse
from tqdm.auto import tqdm

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter


In [16]:


# ============================================================
# 1. Paths and settings
# ============================================================

TEMPLATE_CSV = 'UN_unenv_final.csv'
BASE_PAGE_URL = 'https://unstats.un.org/unsd/envstats/qindicators'
DOWNLOAD_DIR = Path('unsd_env_indicator_files')
OUTPUT_XLSX = 'UNSD_Environmental_Indicators_Metadata_Availability.xlsx'

TIMEOUT = 60

DOWNLOAD_DIR.mkdir(exist_ok=True)

META_COLS = ['no', 'indicator', 'Definition', 'unit', 'source']

In [17]:



# ============================================================
# 2. Load template file
# ============================================================

try:
    template_df = pd.read_csv(TEMPLATE_CSV, encoding='utf-8')
except UnicodeDecodeError:
    template_df = pd.read_csv(TEMPLATE_CSV, encoding='latin1')

country_cols = [col for col in template_df.columns if col not in META_COLS]

print(template_df.head())
print(template_df.shape)
print('Country columns:')
print(len(country_cols))

Empty DataFrame
Columns: [no, indicator, Definition, unit, source, Afghanistan, Albania, Algeria, American Samoa, Andorra, Angola, Antigua and Barbuda, Argentina, Armenia, Aruba, Australia, Austria, Azerbaijan, Bahamas, Bahrain, Bangladesh, Barbados, Belarus, Belgium, Belize, Benin, Bermuda, Bhutan, Bolivia, Bosnia and Herzegovina, Botswana, Brazil, British Virgin Islands, Brunei Darussalam, Bulgaria, Burkina Faso, Burundi, Cabo Verde, Cambodia, Cameroon, Canada, Cayman Islands, Central African Republic, Chad, Channel Islands, Chile, China, Colombia, Comoros, Congo Dem. Rep., Congo Rep., Costa Rica, Côte dIvoire, Croatia, Cuba, Curaçao, Cyprus, Czech Republic, Denmark, Djibouti, Dominica, Dominican Republic, Ecuador, Egypt, El Salvador, Equatorial Guinea, Eritrea, Estonia, Eswatini, Ethiopia, Faroe Islands, Fiji, Finland, France, French Polynesia, Gabon, Gambia, Georgia, Germany, Ghana, Gibraltar, Greece, Greenland, Grenada, Guam, Guatemala, Guinea, Guinea-Bissau, Guyana, Haiti, Hondur

In [18]:



# ============================================================
# 3. Basic helper functions
# ============================================================

def clean_text(value_obj):
    if value_obj is None:
        return ''
    text_val = str(value_obj)
    text_val = text_val.replace('\xa0', ' ')
    text_val = re.sub(r'\s+', ' ', text_val)
    return text_val.strip()


def safe_filename(name_val):
    name_val = unquote(str(name_val))
    name_val = re.sub(r'[<>:"/\\|?*]', '_', name_val)
    name_val = re.sub(r'\s+', ' ', name_val).strip()
    return name_val


def blank_like(value_obj):
    if pd.isna(value_obj):
        return True
    text_val = clean_text(value_obj).lower()
    return text_val in ['', '', 'none']


missing_tokens = {
    '',
    '...',
    '…',
    '..',
    'na',
    '',
    'n/a',
    'none',
    'not available',
    '-'
}


def is_reported_value(value_obj):
    if pd.isna(value_obj):
        return False

    text_val = clean_text(value_obj)

    if text_val.lower() in missing_tokens:
        return False

    return True


def normalize_header(value_obj):
    text_val = clean_text(value_obj)
    text_val = text_val.replace('\n', ' ')
    text_val = re.sub(r'\s+', ' ', text_val).strip()
    return text_val


def is_year_label(value_obj):
    text_val = clean_text(value_obj)

    if re.fullmatch(r'\d{4}', text_val):
        year_val = int(text_val)

        if 1800 <= year_val <= 2100:
            return True

    return False


In [19]:
# ============================================================
# 4. Country-name matching helpers
# ============================================================

def norm_country_name(value_obj):
    text_val = clean_text(value_obj).lower()
    text_val = text_val.replace('&', 'and')
    text_val = text_val.replace('’', "'").replace('`', "'")
    text_val = re.sub(r'[\.,$$$$:;\-/]', ' ', text_val)
    text_val = re.sub(r'\s+', ' ', text_val).strip()
    return text_val


merge_groups = {
    'Bahamas': ['Bahamas', 'Bahamas, The'],
    'Bolivia': ['Bolivia', 'Bolivia (Plurinational State of)', 'Bolivia, Plurinational State of'],
    'British Virgin Islands': ['British Virgin Islands', 'Virgin Islands, British'],
    'Brunei Darussalam': ['Brunei Darussalam', 'Brunei'],
    'Cabo Verde': ['Cabo Verde', 'Cape Verde'],
    'Congo Dem. Rep.': ['Congo Dem. Rep.', 'Dem. Rep. of the Congo', 'Democratic Republic of the Congo', 'Congo, Dem. Rep.'],
    'Congo Rep.': ['Congo Rep.', 'Congo', 'Republic of Congo', 'Congo, Rep.'],
    'Côte dIvoire': ["Côte d'Ivoire", "Cote d'Ivoire", 'Côte dIvoire', 'Côte d’Ivoire'],
    'Curaçao': ['Curaçao', 'Curacao'],
    'Czech Republic': ['Czech Republic', 'Czechia'],
    'Eswatini': ['Eswatini', 'Kingdom of Eswatini', 'Swaziland'],
    'Hong Kong (S.A.R.)': ['Hong Kong (S.A.R.)', 'China, Hong Kong SAR', 'Hong Kong SAR', 'China, Hong Kong Special Administrative Region'],
    'Iran': ['Iran', 'Iran (Islamic Rep. of)', 'Iran (Islamic Republic of)', 'Iran, Islamic Rep.'],
    'Korea Dem. Peoples Rep.': ["Korea Dem. Peoples Rep.", "Democratic People's Republic of Korea", "Korea, Dem.Ppl's.Rep.", 'Korea, Dem. People’s Rep.'],
    'South Korea': ['South Korea', 'Korea', 'Korea, Republic of', 'Republic of Korea'],
    'Lao PDR': ['Lao PDR', "Lao People's Dem. Rep.", "Lao People's Democratic Republic"],
    'Macao SAR': ['Macao SAR', 'China, Macao SAR', 'Macao Special Administrative Region', 'China, Macao Special Administrative Region'],
    'Micronesia': ['Micronesia', 'Micronesia (Fed. States of)', 'Micronesia (Federated States of)', 'Micronesia, Federated States of'],
    'Moldova': ['Moldova', 'Republic of Moldova'],
    'North Macedonia': ['North Macedonia', 'Republic of North Macedonia', 'The former Yugoslav Republic of Macedonia', 'The Former Yugoslav Rep. of Macedonia'],
    'Russia': ['Russia', 'Russian Federation'],
    'Slovak Republic': ['Slovak Republic', 'Slovakia'],
    'Syria': ['Syria', 'Syrian Arab Republic'],
    'Tanzania': ['Tanzania', 'United Rep. of Tanzania', 'United Republic of Tanzania', 'United Republic of Tanzania: Mainland', 'United Republic of Tanzania: Zanzibar'],
    'Turkey': ['Turkey', 'Turkiye', 'Türkiye'],
    'United Kingdom': ['United Kingdom', 'United Kingdom of Great Britain and Northern Ireland', 'United Kingdom of Great Britan and Northern Ireland'],
    'United States': ['United States', 'United States of America', 'USA'],
    'Venezuela': ['Venezuela', 'Venezuela (Bolivarian Republic of)', 'Venezuela, RB'],
    'Vietnam': ['Vietnam', 'Viet Nam'],
    'West Bank and Gaza': ['West Bank and Gaza', 'Palestine', 'State of Palestine']
}

country_lookup = {}

for country_val in country_cols:
    country_lookup[norm_country_name(country_val)] = country_val

for canonical_val, alias_vals in merge_groups.items():
    if canonical_val in country_cols:
        for alias_val in alias_vals:
            country_lookup[norm_country_name(alias_val)] = canonical_val


def country_to_template(value_obj):
    norm_val = norm_country_name(value_obj)

    if norm_val in country_lookup:
        return country_lookup[norm_val]

    norm_val = re.sub(r'^(the )', '', norm_val).strip()

    if norm_val in country_lookup:
        return country_lookup[norm_val]

    return None


print('Country aliases loaded:')
print(len(country_lookup))



Country aliases loaded:
274


In [20]:



# ============================================================
# 5. Create robust web session
# ============================================================

session_obj = requests.Session()

retry_obj = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=1.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=['GET']
)

session_obj.mount('https://', HTTPAdapter(max_retries=retry_obj))
session_obj.mount('http://', HTTPAdapter(max_retries=retry_obj))



In [21]:

# ============================================================
# 6. Scrape UNSD Excel links
# ============================================================

response_obj = session_obj.get(BASE_PAGE_URL, timeout=TIMEOUT)
response_obj.raise_for_status()

soup_obj = BeautifulSoup(response_obj.text, 'html.parser')

link_records = []

for anchor_obj in soup_obj.find_all('a'):
    href_val = anchor_obj.get('href')

    if href_val is None:
        continue

    href_lower = href_val.lower()

    if '.xls' not in href_lower:
        continue

    full_url = urljoin(BASE_PAGE_URL, href_val)

    indicator_text = clean_text(anchor_obj.get_text())

    if indicator_text == '':
        parent_text = clean_text(anchor_obj.parent.get_text()) if anchor_obj.parent is not None else ''
        indicator_text = parent_text

    if indicator_text == '':
        indicator_text = Path(urlparse(full_url).path).stem

    link_records.append({
        'indicator': indicator_text,
        'url': full_url
    })

links_df = pd.DataFrame(link_records).drop_duplicates('url').reset_index(drop=True)

print(links_df.head())
print('Excel links found:')
print(links_df.shape[0])

if links_df.shape[0] == 0:
    raise ValueError('No Excel links were found. The UNSD page layout or URL may have changed.')



                         indicator  \
0                NOx emissions XLS   
1                SO2 emissions XLS   
2                CO2 emissions XLS   
3  CO2 emissions (latest year) XLS   
4     Greenhouse gas emissions XLS   

                                                 url  
0  https://unstats.un.org/unsd/envstats/Questionn...  
1  https://unstats.un.org/unsd/envstats/Questionn...  
2  https://unstats.un.org/unsd/envstats/Questionn...  
3  https://unstats.un.org/unsd/envstats/Questionn...  
4  https://unstats.un.org/unsd/envstats/Questionn...  
Excel links found:
66


In [22]:

# ============================================================
# 7. Download UNSD Excel files
# ============================================================

download_records = []
failed_downloads = []

for record_obj in tqdm(links_df.to_dict('records'), desc='Downloading UNSD workbooks'):
    file_name = safe_filename(Path(urlparse(record_obj['url']).path).name)

    if file_name == '' or '.xls' not in file_name.lower():
        file_name = safe_filename(record_obj['indicator']) + '.xlsx'

    file_path = DOWNLOAD_DIR / file_name

    if not file_path.exists() or file_path.stat().st_size == 0:
        try:
            file_response = session_obj.get(record_obj['url'], timeout=(20, 120))
            file_response.raise_for_status()
            file_path.write_bytes(file_response.content)
            time.sleep(0.2)

        except Exception as exc_obj:
            failed_downloads.append({
                'indicator': record_obj['indicator'],
                'url': record_obj['url'],
                'error': str(exc_obj)[:300]
            })

    if file_path.exists() and file_path.stat().st_size > 0:
        download_records.append({
            'indicator': record_obj['indicator'],
            'url': record_obj['url'],
            'file_path': str(file_path),
            'file_name': file_name
        })

downloads_df = pd.DataFrame(download_records)
failed_downloads_df = pd.DataFrame(failed_downloads)

print(downloads_df.head())
print('Downloaded files:')
print(downloads_df.shape[0])

print('Failed downloads:')
print(failed_downloads_df.shape[0])

if failed_downloads_df.shape[0] > 0:
    print(failed_downloads_df.head(20))



                         indicator  \
0                NOx emissions XLS   
1                SO2 emissions XLS   
2                CO2 emissions XLS   
3  CO2 emissions (latest year) XLS   
4     Greenhouse gas emissions XLS   

                                                 url  \
0  https://unstats.un.org/unsd/envstats/Questionn...   
1  https://unstats.un.org/unsd/envstats/Questionn...   
2  https://unstats.un.org/unsd/envstats/Questionn...   
3  https://unstats.un.org/unsd/envstats/Questionn...   
4  https://unstats.un.org/unsd/envstats/Questionn...   

                                           file_path  \
0        unsd_env_indicator_files\NOx_Emissions.xlsx   
1        unsd_env_indicator_files\SO2_emissions.xlsx   
2  unsd_env_indicator_files\CO2_Emissions_time se...   
3        unsd_env_indicator_files\CO2_Emissions.xlsx   
4        unsd_env_indicator_files\GHG_Emissions.xlsx   

                        file_name  
0              NOx_Emissions.xlsx  
1              SO2_emissi

In [23]:
# ============================================================
# 8. Excel metadata parser
#     Read indicator title, definition, source notes, and unit
#     from the workbook description sheet.
# ============================================================

def get_description_metadata(file_path_obj, fallback_indicator):
    excel_obj = pd.ExcelFile(file_path_obj)

    # --------------------------------------------------------
    # 8A. Find the description sheet
    # --------------------------------------------------------

    desc_sheet = None

    for sheet_val in excel_obj.sheet_names:
        sheet_text = str(sheet_val).lower().strip()

        if sheet_text.startswith('desc'):
            desc_sheet = sheet_val
            break

    if desc_sheet is None:
        desc_sheet = excel_obj.sheet_names[0]


    # --------------------------------------------------------
    # 8B. Read description sheet without headers
    # --------------------------------------------------------

    desc_df = pd.read_excel(
        file_path_obj,
        sheet_name=desc_sheet,
        header=None,
        nrows=200
    )


    # --------------------------------------------------------
    # 8C. Helper to read a metadata value after a label cell
    # --------------------------------------------------------
    # This handles both structures:
    #
    #   Unit:        Number
    #
    # and:
    #
    #   Unit: Number
    #
    # It searches the description sheet cell by cell.

    def extract_value_after_label(label_text):
        label_lower = label_text.lower().strip().rstrip(':')

        for row_idx in range(desc_df.shape[0]):
            for col_idx in range(desc_df.shape[1]):
                cell_val = desc_df.iat[row_idx, col_idx]

                if not is_reported_value(cell_val):
                    continue

                cell_text = clean_text(cell_val)
                cell_lower = cell_text.lower().strip()

                # Case 1: label and value are in the same cell
                # Example: "Unit: Number"
                if cell_lower.startswith(label_lower + ':'):
                    same_cell_value = clean_text(
                        cell_text.split(':', 1)[1]
                    )

                    if same_cell_value != '':
                        return same_cell_value

                # Case 2: label is in one cell and value is to the right
                # Example: "Unit:" | "Number"
                if cell_lower.rstrip(':') == label_lower:
                    right_values = []

                    for next_col_idx in range(col_idx + 1, desc_df.shape[1]):
                        next_val = desc_df.iat[row_idx, next_col_idx]

                        if is_reported_value(next_val):
                            right_values.append(clean_text(next_val))

                    if len(right_values) > 0:
                        return clean_text(' '.join(right_values))

                    # Case 3: value is directly below the label
                    # This is a fallback for unusual layouts.
                    for next_row_idx in range(row_idx + 1, min(row_idx + 4, desc_df.shape[0])):
                        below_val = desc_df.iat[next_row_idx, col_idx]

                        if is_reported_value(below_val):
                            return clean_text(below_val)

        return ''


    # --------------------------------------------------------
    # 8D. Convert non-empty rows to row-level text
    # --------------------------------------------------------

    non_empty_rows = []

    for row_idx in range(desc_df.shape[0]):
        row_vals = []

        for col_idx in range(desc_df.shape[1]):
            val_obj = desc_df.iat[row_idx, col_idx]

            if is_reported_value(val_obj):
                row_vals.append(clean_text(val_obj))

        non_empty_rows.append(row_vals)


    # --------------------------------------------------------
    # 8E. Extract title
    # --------------------------------------------------------

    title_val = fallback_indicator

    for row_vals in non_empty_rows:
        if len(row_vals) == 0:
            continue

        joined_val = clean_text(' '.join(row_vals))
        lower_val = joined_val.lower()

        if joined_val == '':
            continue

        if 'united nations statistics division' in lower_val:
            continue

        if lower_val.startswith('date of release:'):
            continue

        if lower_val.startswith('coverage:'):
            continue

        if lower_val.startswith('series type:'):
            continue

        if lower_val.startswith('unit:'):
            continue

        if lower_val.startswith('definitions'):
            continue

        if lower_val.startswith('data quality'):
            continue

        if lower_val.startswith('source:'):
            continue

        title_val = joined_val
        break


    # --------------------------------------------------------
    # 8F. Extract unit directly from the description sheet
    # --------------------------------------------------------

    unit_val = extract_value_after_label('Unit')


    # --------------------------------------------------------
    # 8G. Extract definition and source notes
    # --------------------------------------------------------

    definition_parts = []
    source_parts = []
    mode_val = None

    for row_vals in non_empty_rows:
        joined_val = clean_text(' '.join(row_vals))
        lower_val = joined_val.lower()

        if joined_val == '':
            continue

        if lower_val.startswith('definitions'):
            mode_val = 'definition'
            continue

        if lower_val.startswith('data quality'):
            mode_val = None
            continue

        if lower_val.startswith('source:'):
            mode_val = 'source'

            source_text = clean_text(joined_val.split(':', 1)[1]) if ':' in joined_val else ''

            if source_text != '':
                source_parts.append(source_text)

            continue

        if lower_val.startswith('related information'):
            mode_val = None
            continue

        if lower_val.startswith('contact:'):
            mode_val = None
            continue

        if mode_val == 'definition':
            definition_parts.append(joined_val)

        if mode_val == 'source':
            source_parts.append(joined_val)

    definition_val = clean_text(' '.join(definition_parts))
    source_notes_val = clean_text(' '.join(source_parts))


    # --------------------------------------------------------
    # 8H. Return metadata
    # --------------------------------------------------------

    return {
        'indicator_title': title_val,
        'Definition': definition_val,
        'unit': unit_val,
        'source_notes': source_notes_val
    }

In [24]:
# ============================================================
# 9. Excel data-sheet parser helpers
# ============================================================

def find_data_sheet(excel_obj):
    for sheet_val in excel_obj.sheet_names:
        if str(sheet_val).lower().strip() == 'data':
            return sheet_val

    for sheet_val in excel_obj.sheet_names:
        if 'data' in str(sheet_val).lower().strip():
            return sheet_val

    return excel_obj.sheet_names[-1]


def make_unique_headers(raw_headers):
    output_vals = []
    counts_obj = {}

    for raw_val in raw_headers:
        header_val = normalize_header(raw_val)

        if header_val == '' or header_val.lower() == '':
            header_val = 'unnamed'

        if header_val in counts_obj:
            counts_obj[header_val] += 1
            header_val = header_val + '_' + str(counts_obj[header_val])
        else:
            counts_obj[header_val] = 0

        output_vals.append(header_val)

    return output_vals


def find_header_row(raw_df):
    best_row = 0
    best_score = -1

    max_rows = min(20, raw_df.shape[0])

    for row_idx in range(max_rows):
        vals = [clean_text(x).lower() for x in raw_df.iloc[row_idx].tolist()]

        has_countryid = any(v == 'countryid' for v in vals)
        has_country = any(
            v in ['country', 'country and area', 'countries or areas']
            for v in vals
        )

        if has_countryid and has_country:
            return row_idx

        if any('latest year available' in v for v in vals) and row_idx <= 2:
            return row_idx

        score = 0

        if has_countryid:
            score += 100

        if has_country:
            score += 100

        score += sum(
            1 for v in vals
            if re.fullmatch(r'\d{4}|\d{4}-\d{4}', v)
        )

        if score > best_score:
            best_score = score
            best_row = row_idx

    return best_row


def make_disaster_headers(raw_df, header_idx):
    main_vals = raw_df.iloc[max(0, header_idx - 1)].tolist()
    sub_vals = raw_df.iloc[header_idx].tolist()

    headers = []
    current_group = ''

    for idx_val, sub_val in enumerate(sub_vals):
        main_val = normalize_header(main_vals[idx_val]) if idx_val < len(main_vals) else ''
        sub_text = normalize_header(sub_val)

        if main_val != '' and main_val.lower() != '':
            current_group = main_val

        if sub_text.lower() in ['countryid', 'country', 'country and area', 'countries or areas']:
            headers.append(sub_text)

        elif re.fullmatch(r'\d{4}-\d{4}', sub_text) and current_group != '':
            headers.append(current_group + ' | ' + sub_text)

        else:
            headers.append(sub_text)

    return make_unique_headers(headers)


def load_data_sheet(file_path_obj):
    excel_obj = pd.ExcelFile(file_path_obj)
    sheet_name_val = find_data_sheet(excel_obj)

    raw_df = pd.read_excel(
        file_path_obj,
        sheet_name=sheet_name_val,
        header=None
    )

    raw_df = (
        raw_df
        .dropna(how='all')
        .dropna(axis=1, how='all')
        .reset_index(drop=True)
    )

    header_idx = find_header_row(raw_df)
    header_vals = raw_df.iloc[header_idx].tolist()

    # Some UNSD files have blank first columns in the header row
    # when "Latest year available" appears in the same row.
    if len(header_vals) >= 2 and blank_like(header_vals[0]) and blank_like(header_vals[1]):
        if any('latest year available' in clean_text(x).lower() for x in header_vals):
            header_vals[0] = 'CountryID'
            header_vals[1] = 'Country'

    # Disaster files often have grouped headers:
    # main header above, period header below.
    if header_idx > 0 and any(
        re.fullmatch(r'\d{4}-\d{4}', clean_text(x))
        for x in header_vals
    ):
        headers = make_disaster_headers(raw_df, header_idx)
    else:
        headers = make_unique_headers(header_vals)

    # The row immediately below the header may contain data-table units.
    # We use this only to determine where data starts.
    # We should not use this row as the final metadata unit.
    units_idx = header_idx + 1 if header_idx + 1 < raw_df.shape[0] else header_idx
    units_row = raw_df.iloc[units_idx].tolist()

    unit_pattern = (
        r'tonne|tonnes|percent|%|kg|m3|km|hectare|hectares|'
        r'number|inhabitant|capita|usd|joule|wh|celsius|mm|mio'
    )

    unit_like = 0

    for x in units_row:
        if blank_like(x):
            unit_like += 1
        elif re.search(unit_pattern, clean_text(x).lower()):
            unit_like += 1

    if unit_like >= 3:
        data_start = header_idx + 2
    else:
        data_start = header_idx + 1

    data_df = raw_df.iloc[data_start:].copy().reset_index(drop=True)
    data_df.columns = headers

    return data_df, units_row, sheet_name_val


def find_country_column(data_df):
    for col_val in data_df.columns:
        low_val = str(col_val).lower().strip()

        if low_val in ['country', 'country and area', 'countries or areas']:
            return col_val

        if low_val.startswith('country '):
            return col_val

    for col_val in data_df.columns:
        low_val = str(col_val).lower().strip()

        if 'country' in low_val or 'countries' in low_val:
            return col_val

    return None


def clean_variable_name(col_val):
    text_val = normalize_header(col_val)
    text_val = re.sub(r'_\d+$', '', text_val)
    return clean_text(text_val)


def unit_for_column(col_idx, units_row, default_unit):
    # IMPORTANT:
    # default_unit should come from the description sheet metadata:
    # the cell labeled "Unit:".
    #
    # The data-sheet unit row is used only as a fallback.
    # This prevents values such as "number" from the data table
    # from replacing the official metadata unit.

    default_unit_clean = clean_text(default_unit)

    if default_unit_clean.lower() not in ['', 'none', 'nan', 'na', 'n/a']:
        return default_unit_clean

    # Fallback only if the description sheet has no Unit value.
    if col_idx < len(units_row):
        unit_val = clean_text(units_row[col_idx])

        if unit_val.lower() not in missing_tokens:
            return unit_val

    return ''


def period_rank(value_obj):
    text_val = clean_text(value_obj)
    found_vals = re.findall(r'\d{4}', text_val)

    if len(found_vals) == 0:
        return 9999

    return max([int(x) for x in found_vals])


def split_variable_period(col_val):
    text_val = clean_variable_name(col_val)

    if ' | ' in text_val:
        left_val, right_val = text_val.rsplit(' | ', 1)
        return clean_text(left_val), clean_text(right_val)

    return text_val, ''


id_cols_lower = set([
    'countryid',
    'country',
    'country and area',
    'countries or areas',
    'source',
    'city',
    'latest year available',
    'year'
])

In [27]:
# ============================================================
# 10. Workbook parser
# ============================================================

def reported_by_country(sub_df, country_col, value_col):
    availability = {country_val: 0 for country_val in country_cols}

    for _, row_obj in sub_df.iterrows():
        template_country = country_to_template(row_obj.get(country_col))

        if template_country is None:
            continue

        if is_reported_value(row_obj.get(value_col)):
            availability[template_country] = 1

    return availability


def parse_workbook(record_obj):
    file_path_obj = Path(record_obj['file_path'])

    # Read workbook-level metadata from the description sheet.
    # The unit should come from the description sheet cell labeled "Unit:".
    meta_obj = get_description_metadata(file_path_obj, record_obj['indicator'])

    # Read the data sheet.
    # units_row is kept only because load_data_sheet returns it.
    # It should not replace the metadata unit in the final output.
    data_df, units_row, sheet_name_val = load_data_sheet(file_path_obj)

    country_col = find_country_column(data_df)

    if country_col is None:
        raise ValueError('Country column not found')

    rows_out = []

    metadata_unit = clean_text(meta_obj.get('unit', ''))


    # --------------------------------------------------------
    # 10A. Annual datasets with year columns
    # --------------------------------------------------------
    # Logic:
    # availability = 1 if a country has at least one reported value
    # in either of the two most recent years.

    year_cols = [
        col_val for col_val in data_df.columns
        if is_year_label(col_val)
    ]

    if len(year_cols) >= 2:
        year_vals = sorted([
            int(clean_text(col_val))
            for col_val in year_cols
        ])

        recent_years = [
            str(year_vals[-2]),
            str(year_vals[-1])
        ]

        availability = {country_val: 0 for country_val in country_cols}

        for _, row_obj in data_df.iterrows():
            template_country = country_to_template(row_obj.get(country_col))

            if template_country is None:
                continue

            has_value = False

            for year_col in recent_years:
                if is_reported_value(row_obj.get(year_col)):
                    has_value = True
                    break

            if has_value:
                availability[template_country] = 1

        row_dict = {
            'indicator': meta_obj['indicator_title'],
            'Definition': meta_obj['Definition'],
            'unit': metadata_unit,
            'source': record_obj['url'],
            '_file_name': record_obj['file_name'],
            '_url': record_obj['url'],
            '_period': max(year_vals),
            '_sheet': sheet_name_val,
            '_variable': meta_obj['indicator_title']
        }

        row_dict.update(availability)
        rows_out.append(row_dict)

        return rows_out


    # --------------------------------------------------------
    # 10B. Datasets with an explicit Year column
    # --------------------------------------------------------
    # Logic:
    # keep only rows from the latest available year before checking
    # availability.

    year_col = None

    for col_val in data_df.columns:
        if str(col_val).lower().strip() == 'year':
            year_col = col_val
            break

    if year_col is not None:
        year_numeric = pd.to_numeric(data_df[year_col], errors='coerce')

        if year_numeric.notna().any():
            latest_period = int(year_numeric.max())
            data_df = data_df.loc[year_numeric == latest_period].copy()
        else:
            latest_period = 9999
    else:
        latest_period = 9999


    # --------------------------------------------------------
    # 10C. Group non-ID variables
    # --------------------------------------------------------
    # For files with multiple variables or period columns, this groups
    # related columns and keeps the most recent period within each group.

    variable_groups = {}

    for idx_val, col_val in enumerate(data_df.columns):
        low_val = str(col_val).lower().strip()

        if low_val in id_cols_lower:
            continue

        if 'footnote' in low_val:
            continue

        if col_val == country_col:
            continue

        variable_name, period_label = split_variable_period(col_val)

        if variable_name.lower() in missing_tokens:
            continue

        if variable_name.lower() == 'unnamed':
            continue

        if variable_name not in variable_groups:
            variable_groups[variable_name] = []

        variable_groups[variable_name].append(
            (
                idx_val,
                col_val,
                period_label,
                period_rank(period_label)
            )
        )


    # --------------------------------------------------------
    # 10D. Create one output row per variable group
    # --------------------------------------------------------

    for variable_name, group_vals in variable_groups.items():
        best_rank = max([x[3] for x in group_vals])
        best_cols = [x for x in group_vals if x[3] == best_rank]

        availability = {country_val: 0 for country_val in country_cols}

        for _, row_obj in data_df.iterrows():
            template_country = country_to_template(row_obj.get(country_col))

            if template_country is None:
                continue

            has_value = False

            for _, col_val, _, _ in best_cols:
                if is_reported_value(row_obj.get(col_val)):
                    has_value = True
                    break

            if has_value:
                availability[template_country] = 1

        if len(variable_groups) == 1:
            indicator_name = meta_obj['indicator_title']
        else:
            indicator_name = variable_name

        row_dict = {
            'indicator': indicator_name,
            'Definition': meta_obj['Definition'],
            'unit': metadata_unit,
            'source': record_obj['url'],
            '_file_name': record_obj['file_name'],
            '_url': record_obj['url'],
            '_period': best_rank if best_rank != 9999 else latest_period,
            '_sheet': sheet_name_val,
            '_variable': variable_name
        }

        row_dict.update(availability)
        rows_out.append(row_dict)

    return rows_out

In [28]:
# ============================================================
# 11. Parse all downloaded workbooks
# ============================================================

output_rows = []
parse_errors = []

for record_obj in tqdm(downloads_df.to_dict('records'), desc='Parsing UNSD workbooks'):
    try:
        workbook_rows = parse_workbook(record_obj)

        if workbook_rows is not None and len(workbook_rows) > 0:
            output_rows.extend(workbook_rows)

    except Exception as exc_obj:
        parse_errors.append({
            'file_name': record_obj.get('file_name'),
            'indicator': record_obj.get('indicator'),
            'error': str(exc_obj)[:300]
        })


# ------------------------------------------------------------
# 11A. Convert parser outputs to DataFrames
# ------------------------------------------------------------

parsed_df = pd.DataFrame(output_rows)
parse_errors_df = pd.DataFrame(parse_errors)


# ------------------------------------------------------------
# 11B. Basic parser summary
# ------------------------------------------------------------

print('Parsed shape:')
print(parsed_df.shape)

print('\nParse errors:')
print(parse_errors_df.shape[0])

if parse_errors_df.shape[0] > 0:
    print('\nParse error examples:')
    display(parse_errors_df.head(20))


# ------------------------------------------------------------
# 11C. Show parsed output preview
# ------------------------------------------------------------

preview_cols = [
    'indicator',
    'unit',
    '_file_name',
    '_period',
    '_sheet',
    '_variable'
]

available_preview_cols = [
    col for col in preview_cols
    if col in parsed_df.columns
]

if parsed_df.shape[0] > 0 and len(available_preview_cols) > 0:
    print('\nParsed output preview:')
    display(parsed_df[available_preview_cols].head(20))
else:
    print('\nNo parsed rows available for preview.')


# ------------------------------------------------------------
# 11D. Check unit extraction from description metadata
# ------------------------------------------------------------

if parsed_df.shape[0] > 0 and 'unit' in parsed_df.columns:
    print('\nUnit value counts:')
    display(
        parsed_df['unit']
        .fillna('')
        .astype(str)
        .str.strip()
        .replace('', '[blank]')
        .value_counts()
        .head(30)
    )

    print('\nRows with blank unit:')
    blank_unit_df = parsed_df[
        parsed_df['unit']
        .fillna('')
        .astype(str)
        .str.strip()
        .eq('')
    ].copy()

    print(blank_unit_df.shape[0])

    if blank_unit_df.shape[0] > 0:
        blank_preview_cols = [
            col for col in ['indicator', 'unit', '_file_name', '_sheet', '_period']
            if col in blank_unit_df.columns
        ]

        display(blank_unit_df[blank_preview_cols].head(30))


# ------------------------------------------------------------
# 11E. Specific check for climatological disaster files
# ------------------------------------------------------------

if parsed_df.shape[0] > 0 and '_file_name' in parsed_df.columns:
    climatological_check_df = parsed_df[
        parsed_df['_file_name']
        .astype(str)
        .str.lower()
        .str.contains('climatological', na=False)
    ].copy()

    print('\nClimatological disaster unit check:')
    print(climatological_check_df.shape[0])

    if climatological_check_df.shape[0] > 0:
        clim_preview_cols = [
            col for col in ['indicator', 'unit', '_file_name', '_sheet', '_period', '_variable']
            if col in climatological_check_df.columns
        ]

        display(
            climatological_check_df[clim_preview_cols]
            .drop_duplicates()
            .head(50)
        )


# ------------------------------------------------------------
# 11F. Stop early if no rows were parsed
# ------------------------------------------------------------

if parsed_df.shape[0] == 0:
    raise ValueError(
        'No parsed rows were produced. Check parse_errors_df and the downloaded workbook files.'
    )

Parsing UNSD workbooks:   0%|          | 0/66 [00:00<?, ?it/s]

Parsed shape:
(171, 227)

Parse errors:
0

Parsed output preview:


,indicator,unit,_file_name,_period,_sheet,_variable
0,NOx emissions,thousand tonnes (1000 tonnes) or kilogrammes (...,NOx_Emissions.xlsx,9999,Data,NOx emissions
1,% change since 1990,thousand tonnes (1000 tonnes) or kilogrammes (...,NOx_Emissions.xlsx,9999,Data,% change since 1990
2,NOx emissions per capita,thousand tonnes (1000 tonnes) or kilogrammes (...,NOx_Emissions.xlsx,9999,Data,NOx emissions per capita
3,SO2 emissions,Thousand tonnes (1000 tonnes) or kilogrammes (...,SO2_emissions.xlsx,9999,Data,SO2 emissions
4,%change since 1990,Thousand tonnes (1000 tonnes) or kilogrammes (...,SO2_emissions.xlsx,9999,Data,%change since 1990
5,SO2 emissions per capita,Thousand tonnes (1000 tonnes) or kilogrammes (...,SO2_emissions.xlsx,9999,Data,SO2 emissions per capita
6,Total CO2 emisions,Thous. tonnes of CO2 equivalent,CO2_Emissions_time series.xlsx,2021,Data,Total CO2 emisions
7,CO2 emissions,mio. tonnes of CO2 equivalent or tonnes of CO2...,CO2_Emissions.xlsx,9999,Data,CO2 emissions
8,% change since 1990,mio. tonnes of CO2 equivalent or tonnes of CO2...,CO2_Emissions.xlsx,9999,Data,% change since 1990
9,CO2 emissions per capita,mio. tonnes of CO2 equivalent or tonnes of CO2...,CO2_Emissions.xlsx,9999,Data,CO2 emissions per capita



Unit value counts:


unit
Number of occurrences and number of people                                                                  16
mio. tonnes of CO2 equivalent or tonnes of CO2 equivalent or percentage (%), as indicated                   15
Year                                                                                                        14
Thousands of tonnes (1000 tonnes) or percentage (%) as indicated                                            12
Percentage                                                                                                  12
Million cubic metres                                                                                        10
1000 ha, or percentage, as indicated                                                                         9
Square kilometers (km2) or percentage (%) as indicated                                                       8
Million cubic metres (mio m3) or cubic metres (m3) per capita as indicated                                 


Rows with blank unit:
4


,indicator,unit,_file_name,_sheet,_period
111,nan,,Consumption of fertilizers per unit of agricul...,2013,9999
112,Nitrogen,,Consumption of fertilizers per unit of agricul...,2013,9999
113,Phosphate,,Consumption of fertilizers per unit of agricul...,2013,9999
114,Potash,,Consumption of fertilizers per unit of agricul...,2013,9999



Climatological disaster unit check:
4


,indicator,unit,_file_name,_sheet,_period,_variable
116,Occurrence,Number of occurrences and number of people,Climatological disasters.xlsx,data,2003,Occurrence
117,nan,Number of occurrences and number of people,Climatological disasters.xlsx,data,2023,nan
118,Total deaths,Number of occurrences and number of people,Climatological disasters.xlsx,data,2003,Total deaths
119,Persons affected,Number of occurrences and number of people,Climatological disasters.xlsx,data,2003,Persons affected


In [29]:
# ============================================================
# 12. Final cleanup
#     - remove blank indicator rows
#     - preserve metadata unit from description sheet
#     - create helper columns for deduplication
#     - remove older duplicate year rows
#     - build final output using the template column structure
# ============================================================

if parsed_df.shape[0] == 0:
    raise ValueError('No parsed rows found. Check downloads and parser errors.')

parsed_df = parsed_df.copy()


# ------------------------------------------------------------
# 12A. Check required parser columns
# ------------------------------------------------------------

minimum_parser_cols = ['indicator', 'Definition', 'unit', 'source']

missing_parser_cols = [
    col for col in minimum_parser_cols
    if col not in parsed_df.columns
]

if len(missing_parser_cols) > 0:
    raise KeyError(
        'The following required parser columns are missing from parsed_df: '
        + ', '.join(missing_parser_cols)
    )


# ------------------------------------------------------------
# 12B. Clean core metadata columns
# ------------------------------------------------------------

for text_col in ['indicator', 'Definition', 'unit', 'source']:
    parsed_df[text_col] = (
        parsed_df[text_col]
        .fillna('')
        .astype(str)
        .str.replace('\xa0', ' ', regex=False)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )


# ------------------------------------------------------------
# 12C. Remove blank indicator rows
# ------------------------------------------------------------

parsed_df = parsed_df[
    ~parsed_df['indicator'].str.lower().isin(['', 'none', 'nan', 'na', 'n/a'])
].copy()


# ------------------------------------------------------------
# 12D. Helper function to remove trailing year from indicator
# ------------------------------------------------------------
# Example:
# Forest Area_c_Forest Area, 1990 -> Forest Area_c_Forest Area
# Forest Area_c_Forest Area, 2020 -> Forest Area_c_Forest Area

def get_base_indicator(indicator_value):
    text_val = clean_text(indicator_value)
    text_val = re.sub(r'\s*,\s*((18|19|20|21)\d{2})\s*$', '', text_val)
    return text_val.strip()


# ------------------------------------------------------------
# 12E. Helper function to extract trailing year from indicator
# ------------------------------------------------------------
# Example:
# Forest Area_c_Forest Area, 2020 -> 2020

def get_indicator_year(indicator_value):
    text_val = clean_text(indicator_value)
    match_obj = re.search(r'\s*,\s*((18|19|20|21)\d{2})\s*$', text_val)

    if match_obj:
        return int(match_obj.group(1))

    return np.nan


# ------------------------------------------------------------
# 12F. Create helper columns for deduplication
# ------------------------------------------------------------

parsed_df['_base_indicator_for_dedup'] = parsed_df['indicator'].apply(get_base_indicator)
parsed_df['_indicator_year_for_dedup'] = parsed_df['indicator'].apply(get_indicator_year)

print('Deduplication helper columns created:')

dedup_preview_cols = [
    col for col in [
        'indicator',
        'unit',
        '_file_name',
        '_period',
        '_base_indicator_for_dedup',
        '_indicator_year_for_dedup'
    ]
    if col in parsed_df.columns
]

display(parsed_df[dedup_preview_cols].head(10))


# ------------------------------------------------------------
# 12G. Split rows into year-based rows and non-year rows
# ------------------------------------------------------------

year_rows_df = parsed_df[
    parsed_df['_indicator_year_for_dedup'].notna()
].copy()

non_year_rows_df = parsed_df[
    parsed_df['_indicator_year_for_dedup'].isna()
].copy()

print('Rows with trailing indicator year:')
print(year_rows_df.shape[0])

print('Rows without trailing indicator year:')
print(non_year_rows_df.shape[0])


# ------------------------------------------------------------
# 12H. For duplicate year rows, keep only the most recent year
# ------------------------------------------------------------
# Duplicate definition:
# same source file + same base indicator
#
# Example:
# Forest Area_c_Forest Area, 1990
# Forest Area_c_Forest Area, 2000
# Forest Area_c_Forest Area, 2010
# Forest Area_c_Forest Area, 2015
# Forest Area_c_Forest Area, 2020
#
# becomes:
# Forest Area_c_Forest Area, 2020

if year_rows_df.shape[0] > 0:
    sort_cols_year = [
        col for col in [
            '_file_name',
            '_base_indicator_for_dedup',
            '_indicator_year_for_dedup'
        ]
        if col in year_rows_df.columns
    ]

    if len(sort_cols_year) > 0:
        year_rows_df = year_rows_df.sort_values(sort_cols_year)

    dedup_cols_year = [
        col for col in [
            '_file_name',
            '_base_indicator_for_dedup'
        ]
        if col in year_rows_df.columns
    ]

    if len(dedup_cols_year) == 0:
        dedup_cols_year = ['indicator']

    year_rows_df = (
        year_rows_df
        .drop_duplicates(dedup_cols_year, keep='last')
        .copy()
    )


# ------------------------------------------------------------
# 12I. For non-year rows, remove duplicate indicator rows
# ------------------------------------------------------------

if non_year_rows_df.shape[0] > 0:
    dedup_cols_non_year = []

    if '_file_name' in non_year_rows_df.columns:
        dedup_cols_non_year.append('_file_name')

    dedup_cols_non_year.append('indicator')

    sort_cols_non_year = []

    if '_file_name' in non_year_rows_df.columns:
        sort_cols_non_year.append('_file_name')

    if '_variable' in non_year_rows_df.columns:
        sort_cols_non_year.append('_variable')

    if '_period' in non_year_rows_df.columns:
        sort_cols_non_year.append('_period')

    if len(sort_cols_non_year) > 0:
        non_year_rows_df = non_year_rows_df.sort_values(sort_cols_non_year)

    non_year_rows_df = (
        non_year_rows_df
        .drop_duplicates(dedup_cols_non_year, keep='last')
        .copy()
    )


# ------------------------------------------------------------
# 12J. Combine cleaned rows back together
# ------------------------------------------------------------

parsed_df = pd.concat(
    [non_year_rows_df, year_rows_df],
    ignore_index=True
)

sort_cols_final = []

if '_file_name' in parsed_df.columns:
    sort_cols_final.append('_file_name')

if '_base_indicator_for_dedup' in parsed_df.columns:
    sort_cols_final.append('_base_indicator_for_dedup')

if '_indicator_year_for_dedup' in parsed_df.columns:
    sort_cols_final.append('_indicator_year_for_dedup')

if len(sort_cols_final) > 0:
    parsed_df = parsed_df.sort_values(
        sort_cols_final,
        na_position='last'
    ).reset_index(drop=True)


# ------------------------------------------------------------
# 12K. Check required output columns
# ------------------------------------------------------------

required_output_cols = ['indicator', 'Definition', 'unit', 'source'] + country_cols

missing_output_cols = [
    col for col in required_output_cols
    if col not in parsed_df.columns
]

if len(missing_output_cols) > 0:
    raise KeyError(
        'The following required columns are missing from parsed_df: '
        + ', '.join(missing_output_cols)
    )


# ------------------------------------------------------------
# 12L. Build final output using template column structure
# ------------------------------------------------------------

final_df = parsed_df[required_output_cols].copy()

final_df.insert(0, 'no', range(1, len(final_df) + 1))


# ------------------------------------------------------------
# 12M. Final Step 12 diagnostics
# ------------------------------------------------------------

print('Final output preview:')
display(final_df.head())

print('Final output shape:')
print(final_df.shape)

print('Unit value counts in final output:')
display(
    final_df['unit']
    .fillna('')
    .astype(str)
    .str.strip()
    .replace('', '[blank]')
    .value_counts()
    .head(30)
)

Deduplication helper columns created:


,indicator,unit,_file_name,_period,_base_indicator_for_dedup,_indicator_year_for_dedup
0,NOx emissions,thousand tonnes (1000 tonnes) or kilogrammes (...,NOx_Emissions.xlsx,9999,NOx emissions,NaN
1,% change since 1990,thousand tonnes (1000 tonnes) or kilogrammes (...,NOx_Emissions.xlsx,9999,% change since 1990,NaN
2,NOx emissions per capita,thousand tonnes (1000 tonnes) or kilogrammes (...,NOx_Emissions.xlsx,9999,NOx emissions per capita,NaN
3,SO2 emissions,Thousand tonnes (1000 tonnes) or kilogrammes (...,SO2_emissions.xlsx,9999,SO2 emissions,NaN
4,%change since 1990,Thousand tonnes (1000 tonnes) or kilogrammes (...,SO2_emissions.xlsx,9999,%change since 1990,NaN
5,SO2 emissions per capita,Thousand tonnes (1000 tonnes) or kilogrammes (...,SO2_emissions.xlsx,9999,SO2 emissions per capita,NaN
6,Total CO2 emisions,Thous. tonnes of CO2 equivalent,CO2_Emissions_time series.xlsx,2021,Total CO2 emisions,NaN
7,CO2 emissions,mio. tonnes of CO2 equivalent or tonnes of CO2...,CO2_Emissions.xlsx,9999,CO2 emissions,NaN
8,% change since 1990,mio. tonnes of CO2 equivalent or tonnes of CO2...,CO2_Emissions.xlsx,9999,% change since 1990,NaN
9,CO2 emissions per capita,mio. tonnes of CO2 equivalent or tonnes of CO2...,CO2_Emissions.xlsx,9999,CO2 emissions per capita,NaN


Rows with trailing indicator year:
8
Rows without trailing indicator year:
154
Final output preview:


,no,indicator,Definition,unit,source,Afghanistan,Albania,Algeria,American Samoa,Andorra,...,Uruguay,Uzbekistan,Vanuatu,Venezuela,Viet Nam,Virgin Islands (U.S.),West Bank and Gaza,Yemen,Zambia,Zimbabwe
0,1,Actual evapotranspiration,Actual evapotranspiration: Total actual volume...,Million cubic metres,https://unstats.un.org/unsd/envstats/Questionn...,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,2,% change of agricultural area since 1990,Agricultural area is the sum of areas under (a...,Square kilometers (km2) or percentage (%) as i...,https://unstats.un.org/unsd/envstats/Questionn...,1,1,1,1,1,...,1,0,1,1,1,0,0,1,1,1
2,3,% of total land area covered by agricultural a...,Agricultural area is the sum of areas under (a...,Square kilometers (km2) or percentage (%) as i...,https://unstats.un.org/unsd/envstats/Questionn...,1,1,1,1,1,...,1,1,1,1,1,0,1,1,1,1
3,4,Agricultural area actually irrigated in 2021,Agricultural area is the sum of areas under (a...,Square kilometers (km2) or percentage (%) as i...,https://unstats.un.org/unsd/envstats/Questionn...,1,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,5,Agricultural area in 2021,Agricultural area is the sum of areas under (a...,Square kilometers (km2) or percentage (%) as i...,https://unstats.un.org/unsd/envstats/Questionn...,1,1,1,1,1,...,1,1,1,1,1,0,1,1,1,1


Final output shape:
(158, 223)
Unit value counts in final output:


unit
Year                                                                                                        14
mio. tonnes of CO2 equivalent or tonnes of CO2 equivalent or percentage (%), as indicated                   13
Percentage                                                                                                  12
Thousands of tonnes (1000 tonnes) or percentage (%) as indicated                                            12
Number of occurrences and number of people                                                                  12
Million cubic metres                                                                                        10
Square kilometers (km2) or percentage (%) as indicated                                                       7
mio. tonnes of CO2 equivalent                                                                                7
Million cubic metres (mio m3) or cubic metres (m3) per capita as indicated                                 

In [31]:
 # ============================================================
# 13. Export to Excel
# ============================================================

# Export final DataFrame
final_df.to_excel(OUTPUT_XLSX, index=False)

# Load workbook for formatting
wb_obj = load_workbook(OUTPUT_XLSX)
ws_obj = wb_obj.active
ws_obj.title = 'Availability'


# ------------------------------------------------------------
# 13A. Add hyperlinks to the source column
# ------------------------------------------------------------

# Column positions:
# no = 1
# indicator = 2
# Definition = 3
# unit = 4
# source = 5

source_col_idx = 5

for row_idx in range(2, ws_obj.max_row + 1):
    cell_obj = ws_obj.cell(row=row_idx, column=source_col_idx)

    if cell_obj.value:
        cell_obj.hyperlink = str(cell_obj.value)
        cell_obj.style = 'Hyperlink'


# ------------------------------------------------------------
# 13B. Format header row
# ------------------------------------------------------------

for col_idx in range(1, ws_obj.max_column + 1):
    header_cell = ws_obj.cell(row=1, column=col_idx)
    header_cell.font = Font(bold=True)
    header_cell.alignment = Alignment(
        wrap_text=True,
        horizontal='center',
        vertical='center'
    )


# ------------------------------------------------------------
# 13C. Format all cells
# ------------------------------------------------------------

for row in ws_obj.iter_rows():
    for cell in row:
        cell.alignment = Alignment(
            wrap_text=True,
            vertical='top'
        )


# ------------------------------------------------------------
# 13D. Adjust column widths
# ------------------------------------------------------------

for col_idx in range(1, ws_obj.max_column + 1):
    col_letter = get_column_letter(col_idx)

    if col_idx == 1:
        ws_obj.column_dimensions[col_letter].width = 8

    elif col_idx == 2:
        ws_obj.column_dimensions[col_letter].width = 40

    elif col_idx == 3:
        ws_obj.column_dimensions[col_letter].width = 60

    elif col_idx == 4:
        ws_obj.column_dimensions[col_letter].width = 25

    elif col_idx == 5:
        ws_obj.column_dimensions[col_letter].width = 45

    else:
        ws_obj.column_dimensions[col_letter].width = 16


# ------------------------------------------------------------
# 13E. Freeze panes and add autofilter
# ------------------------------------------------------------

ws_obj.freeze_panes = 'F2'
ws_obj.auto_filter.ref = ws_obj.dimensions


# ------------------------------------------------------------
# 13F. Save workbook
# ------------------------------------------------------------

wb_obj.save(OUTPUT_XLSX)

print('Saved final workbook:')
print(OUTPUT_XLSX)

Saved final workbook:
UNSD_Environmental_Indicators_Metadata_Availability.xlsx


In [32]:
 # ============================================================
# 14. Final quality checks
# ============================================================

print('============================================================')
print('FINAL QUALITY CHECKS')
print('============================================================')

print('Final output shape:')
print(final_df.shape)

print('\nNumber of indicator rows:')
print(final_df.shape[0])

print('\nNumber of country columns:')
print(len(country_cols))

print('\nFailed downloads:')
print(failed_downloads_df.shape[0])

if failed_downloads_df.shape[0] > 0:
    print('\nFailed download examples:')
    display(failed_downloads_df.head(20))

print('\nParse errors:')
print(parse_errors_df.shape[0])

if parse_errors_df.shape[0] > 0:
    print('\nParse error examples:')
    display(parse_errors_df.head(20))


# ------------------------------------------------------------
# 14A. Availability summary per indicator
# ------------------------------------------------------------

country_availability_counts = final_df[country_cols].sum(axis=1)

print('\nAvailability summary per indicator:')
print(country_availability_counts.describe())


# ------------------------------------------------------------
# 14B. Indicators with zero country availability
# ------------------------------------------------------------

zero_availability_df = final_df.loc[
    country_availability_counts == 0,
    ['no', 'indicator', 'unit', 'source']
].copy()

print('\nIndicators with zero country availability:')
print(zero_availability_df.shape[0])

if zero_availability_df.shape[0] > 0:
    display(zero_availability_df.head(20))


# ------------------------------------------------------------
# 14C. Country-level availability summary
# ------------------------------------------------------------

country_summary_df = pd.DataFrame({
    'country': country_cols,
    'available_indicators': final_df[country_cols].sum(axis=0).values
})

country_summary_df['total_indicators'] = final_df.shape[0]
country_summary_df['availability_rate_percent'] = (
    country_summary_df['available_indicators']
    / country_summary_df['total_indicators']
    * 100
).round(2)

country_summary_df = country_summary_df.sort_values(
    'available_indicators',
    ascending=False
).reset_index(drop=True)

print('\nCountry-level availability summary:')
display(country_summary_df.head(20))


# ------------------------------------------------------------
# 14D. Save country-level summary as a separate Excel file
# ------------------------------------------------------------

COUNTRY_SUMMARY_XLSX = 'UNSD_Country_Availability_Summary.xlsx'

country_summary_df.to_excel(COUNTRY_SUMMARY_XLSX, index=False)

print('\nSaved country-level summary:')
print(COUNTRY_SUMMARY_XLSX)


# ------------------------------------------------------------
# 14E. Save parse errors and failed downloads, if any
# ------------------------------------------------------------

if failed_downloads_df.shape[0] > 0:
    FAILED_DOWNLOADS_XLSX = 'UNSD_Failed_Downloads.xlsx'
    failed_downloads_df.to_excel(FAILED_DOWNLOADS_XLSX, index=False)

    print('\nSaved failed downloads file:')
    print(FAILED_DOWNLOADS_XLSX)

if parse_errors_df.shape[0] > 0:
    PARSE_ERRORS_XLSX = 'UNSD_Parse_Errors.xlsx'
    parse_errors_df.to_excel(PARSE_ERRORS_XLSX, index=False)

    print('\nSaved parse errors file:')
    print(PARSE_ERRORS_XLSX)


print('\nDone.')

FINAL QUALITY CHECKS
Final output shape:
(158, 223)

Number of indicator rows:
158

Number of country columns:
218

Failed downloads:
0

Parse errors:
0

Availability summary per indicator:
count    158.000000
mean      98.613924
std       69.013035
min        0.000000
25%       39.250000
50%       82.500000
75%      175.750000
max      205.000000
dtype: float64

Indicators with zero country availability:
5


,no,indicator,unit,source
42,43,Freshwater abstracted,Million cubic metres (mio m3) or percentage (%...,https://unstats.un.org/unsd/envstats/Questionn...
43,44,Freshwater abstracted as a proportion of renew...,Million cubic metres (mio m3) or percentage (%...,https://unstats.un.org/unsd/envstats/Questionn...
44,45,Renewable freshwater resources,Million cubic metres (mio m3) or percentage (%...,https://unstats.un.org/unsd/envstats/Questionn...
45,46,latest year available,Million cubic metres (mio m3) or percentage (%...,https://unstats.un.org/unsd/envstats/Questionn...
137,138,Total population supplied by water supply indu...,Percentage,https://unstats.un.org/unsd/envstats/Questionn...



Country-level availability summary:


,country,available_indicators,total_indicators,availability_rate_percent
0,Kazakhstan,136,158,86.08
1,Belarus,135,158,85.44
2,Armenia,126,158,79.75
3,Azerbaijan,118,158,74.68
4,Oman,117,158,74.05
5,Croatia,116,158,73.42
6,Hungary,109,158,68.99
7,Poland,109,158,68.99
8,Spain,107,158,67.72
9,Cuba,107,158,67.72



Saved country-level summary:
UNSD_Country_Availability_Summary.xlsx

Done.


In [33]:
 # ============================================================
# 15. Optional: Save final output, country summary, and errors
#     in one workbook with multiple sheets
# ============================================================

MULTISHEET_OUTPUT_XLSX = 'UNSD_Environmental_Indicators_Final_Output.xlsx'

with pd.ExcelWriter(MULTISHEET_OUTPUT_XLSX, engine='openpyxl') as writer:
    final_df.to_excel(writer, sheet_name='Availability', index=False)
    country_summary_df.to_excel(writer, sheet_name='Country summary', index=False)

    if failed_downloads_df.shape[0] > 0:
        failed_downloads_df.to_excel(writer, sheet_name='Failed downloads', index=False)

    if parse_errors_df.shape[0] > 0:
        parse_errors_df.to_excel(writer, sheet_name='Parse errors', index=False)

print('Saved multisheet workbook:')
print(MULTISHEET_OUTPUT_XLSX)

Saved multisheet workbook:
UNSD_Environmental_Indicators_Final_Output.xlsx
